<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#라이브러리
import pandas as pd
import re
from urllib.parse import urlparse
from google.colab import drive
import requests

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**피싱 URL 데이터**

In [ ]:
#데이터셋 로드
df = pd.read_csv('/content/drive/MyDrive/malicious_phish.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================
# Label 생성
# ==========================

df["label"] = df["type"].apply(
    lambda x: 0 if x == "benign" else 1
)

# ==========================
# URL Parsing
# ==========================

def parse_url(url):
    try:
        if not str(url).startswith(("http://", "https://")):
            url = "http://" + str(url)

        parsed = urlparse(url)

        return pd.Series({
            "scheme": parsed.scheme,
            "domain": parsed.netloc,
            "path": parsed.path,
            "query": parsed.query
        })
    except ValueError:
        # Handle the error by returning empty values for malformed URLs
        return pd.Series({
            "scheme": "",
            "domain": "",
            "path": "",
            "query": ""
        })

parsed = df["url"].apply(parse_url)

df = pd.concat([df, parsed], axis=1)

# ==========================
# URL Length
# ==========================

df["url_length"] = df["url"].astype(str).apply(len)

# ==========================
# Domain Length
# ==========================

df["domain_length"] = df["domain"].astype(str).apply(len)

# ==========================
# Dot Count
# ==========================

df["dot_count"] = df["url"].astype(str).str.count(r"\.")

# ==========================
# Slash Count
# ==========================

df["slash_count"] = df["url"].astype(str).str.count("/")

# ==========================
# Hyphen Count
# ==========================

df["hyphen_count"] = df["url"].astype(str).str.count("-")

# ==========================
# Digit Count
# ==========================

df["digit_count"] = df["url"].astype(str).apply(
    lambda x: len(re.findall(r"\d", x))
)

# ==========================
# @ 포함 여부
# ==========================

df["contains_at"] = df["url"].astype(str).apply(
    lambda x: 1 if "@" in x else 0
)

# ==========================
# IP 사용 여부
# ==========================

def has_ip(url):
    pattern = r"(?:\d{1,3}\.){3}\d{1,3}"
    return 1 if re.search(pattern, str(url)) else 0

df["has_ip"] = df["url"].apply(has_ip)

# ==========================
# HTTPS 여부
# ==========================

df["https_flag"] = (
    df["scheme"]
    .apply(lambda x: 1 if x == "https" else 0)
)

# ==========================
# 단축 URL
# ==========================

shorteners = [
    "bit.ly",
    "tinyurl",
    "goo.gl",
    "t.co"
]

df["shortener"] = df["url"].apply(
    lambda x: int(
        any(
            s in str(x).lower()
            for s in shorteners
        )
    )
)

# ==========================
# 의심 키워드
# ==========================

keywords = [
    "login",
    "verify",
    "account",
    "secure",
    "update",
    "bank",
    "paypal",
    "apple",
    "facebook"
]

def suspicious_count(url):
    return sum(
        word in str(url).lower()
        for word in keywords
    )

df["suspicious_word_count"] = (
    df["url"].apply(suspicious_count)
)

# 저장
save_path = "/content/drive/MyDrive/REDRED/url_features.csv"
df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print(df.head())

                                                 url        type  label  \
0                                   br-icloud.com.br    phishing      1   
1                mp3raid.com/music/krizz_kaliko.html      benign      0   
2                    bopsecrets.org/rexroth/cr/1.htm      benign      0   
3  http://www.garage-pirenne.be/index.php?option=...  defacement      1   
4  http://adventure-nicaragua.net/index.php?optio...  defacement      1   

  scheme                   domain                      path  \
0   http         br-icloud.com.br                             
1   http              mp3raid.com  /music/krizz_kaliko.html   
2   http           bopsecrets.org         /rexroth/cr/1.htm   
3   http    www.garage-pirenne.be                /index.php   
4   http  adventure-nicaragua.net                /index.php   

                                               query  url_length  \
0                                                             16   
1                                 

**재난 안전 문자 API**

In [7]:
import requests
import pandas as pd
import time

SERVICE_KEY = "API 키"

url = "https://www.safetydata.go.kr/V2/api/DSSP-IF-00247"

all_messages = []
seen_sn = set()

page = 1

while True:

    params = {
        "serviceKey": SERVICE_KEY,
        "pageNo": page,
        "numOfRows": 1000,
        "returnType": "json",
        "crtDt": "20240101"
    }

    response = requests.get(url, params=params)
    data = response.json()

    header = data.get("header", {})

    if header.get("resultCode") == "22":
        print("호출 제한 발생 → 60초 대기")
        time.sleep(60)
        continue

    body = data.get("body")

    if not body:
        print("데이터 없음")
        break

    new_count = 0

    for item in body:

        sn = item["SN"]

        if sn not in seen_sn:
            seen_sn.add(sn)
            all_messages.append(item)
            new_count += 1

    print(
        f"page={page} "
        f"수집={len(body)} "
        f"신규={new_count}"
    )

    if new_count == 0:
        print("신규 데이터 없음 → 종료")
        break

    page += 1

    time.sleep(1)

df = pd.DataFrame(all_messages)

df["CRT_DT"] = pd.to_datetime(df["CRT_DT"])

df_2024 = df[
    df["CRT_DT"].dt.year == 2024
]

print("2024년 데이터 수:", len(df_2024))

page=1 수집=1000 신규=1000
page=2 수집=1000 신규=1000
page=3 수집=1000 신규=1000
page=4 수집=1000 신규=1000
page=5 수집=1000 신규=1000
page=6 수집=1000 신규=1000
page=7 수집=1000 신규=1000
page=8 수집=1000 신규=1000
page=9 수집=1000 신규=1000
page=10 수집=1000 신규=1000
page=11 수집=1000 신규=1000
page=12 수집=1000 신규=1000
page=13 수집=1000 신규=1000
page=14 수집=1000 신규=1000
page=15 수집=1000 신규=1000
page=16 수집=1000 신규=1000
page=17 수집=1000 신규=1000
page=18 수집=1000 신규=1000
page=19 수집=1000 신규=1000
page=20 수집=1000 신규=1000
page=21 수집=1000 신규=1000
page=22 수집=1000 신규=1000
page=23 수집=1000 신규=1000
page=24 수집=1000 신규=1000
page=25 수집=1000 신규=1000
page=26 수집=1000 신규=1000
page=27 수집=1000 신규=1000
page=28 수집=1000 신규=1000
page=29 수집=1000 신규=1000
page=30 수집=1000 신규=1000
page=31 수집=1000 신규=1000
page=32 수집=1000 신규=1000
page=33 수집=1000 신규=1000
page=34 수집=1000 신규=1000
page=35 수집=1000 신규=1000
page=36 수집=1000 신규=1000
page=37 수집=1000 신규=1000
page=38 수집=1000 신규=1000
page=39 수집=1000 신규=1000
page=40 수집=1000 신규=1000
page=41 수집=1000 신규=1000
page=42 수집=1000 신규=1000
p

In [9]:
import requests
import pandas as pd
import re

# ==========================================
# 2. DataFrame 생성
# ==========================================

df = df_2024.copy()

df = df.rename(columns={
    "MSG_CN": "message",
    "RCPTN_RGN_NM": "region",
    "DST_SE_NM": "disaster_type",
    "EMRG_STEP_NM": "alert_type"
})

# ==========================================
# 3. 기관명 추출
# ==========================================

df["agency"] = df["message"].str.extract(
    r"\[([^\]]+)\]"
)

# ==========================================
# 4. 기관명 제거
# ==========================================

df["clean_text"] = df["message"].str.replace(
    r"\[[^\]]+\]",
    "",
    regex=True
)

# ==========================================
# 5. URL 추출
# ==========================================

url_pattern = r'https?://[^\s]+|www\.[^\s]+'

df["urls"] = df["message"].apply(
    lambda x: re.findall(url_pattern, str(x))
)

# URL 존재 여부

df["has_url"] = df["urls"].apply(
    lambda x: 1 if len(x) > 0 else 0
)

# URL 개수

df["url_count"] = df["urls"].apply(len)

# ==========================================
# 6. 텍스트 길이
# ==========================================

df["text_length"] = df["clean_text"].str.len()

# ==========================================
# 7. 재난 키워드 Feature
# ==========================================

disaster_keywords = [
    "호우",
    "태풍",
    "폭염",
    "한파",
    "산불",
    "화재",
    "강풍",
    "지진",
    "홍수",
    "붕괴",
    "대설",
    "황사"
]

def disaster_keyword_count(text):
    return sum(
        keyword in str(text)
        for keyword in disaster_keywords
    )

df["disaster_keyword_count"] = (
    df["clean_text"]
    .apply(disaster_keyword_count)
)

# ==========================================
# 8. 스미싱 키워드 Feature
# ==========================================

smishing_keywords = [
    "지원금",
    "보상금",
    "당첨",
    "쿠폰",
    "수령",
    "환급",
    "조회",
    "확인",
    "상품권",
    "이벤트",
    "링크",
    "접속",
    "인증",
    "로그인",
    "계정",
    "결제"
]

def smishing_keyword_count(text):
    return sum(
        keyword in str(text)
        for keyword in smishing_keywords
    )

df["smishing_keyword_count"] = (
    df["clean_text"]
    .apply(smishing_keyword_count)
)

# ==========================================
# 9. 긴급성 키워드 Feature
# ==========================================

emergency_keywords = [
    "긴급",
    "즉시",
    "지금",
    "오늘",
    "신속",
    "필수",
    "주의",
    "경보",
    "발효"
]

def emergency_keyword_count(text):
    return sum(
        keyword in str(text)
        for keyword in emergency_keywords
    )

df["emergency_keyword_count"] = (
    df["clean_text"]
    .apply(emergency_keyword_count)
)

# ==========================================
# 10. 공백 정리
# ==========================================

df["clean_text"] = (
    df["clean_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ==========================================
# 11. 최종 컬럼 확인
# ==========================================

df["CRT_DT"] = pd.to_datetime(df["CRT_DT"])

df["date"] = df["CRT_DT"].dt.date
df["hour"] = df["CRT_DT"].dt.hour

final_df = df[
    [
        "date",
        "hour",
        "message",
        "clean_text",
        "agency",
        "region",
        "disaster_type",
        "alert_type",
        "has_url",
        "url_count",
        "text_length",
        "disaster_keyword_count",
        "smishing_keyword_count",
        "emergency_keyword_count"
    ]
]

print(final_df.head())

# ==========================================
# 12. CSV 저장
# ==========================================

save_path = "/content/drive/MyDrive/REDRED/disaster_sms_preprocessed.csv"
final_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

         date  hour                                            message  \
0  2024-01-10     7  밤사이 내린 눈과 비로 도로와 인도의 결빙이 우려되오니 감속운행, 대중교통 이용, ...   
1  2024-01-10     7  상주지역에 새벽까지 내린 눈으로 도로결빙이 우려되오니 차량운행시 감속 등 안전운전하...   
2  2024-01-10     7  밤새 내린 눈과 낮은기온으로 인하여 도로 결빙이 우려되오니 차량운행(감속운행, 차간...   
3  2024-01-11    11  북구에서 실종된 김균석씨(남,60세)를 찾습니다 -150cm,85kg,비만체격,흰색...   
4  2024-05-06     5  전일부터 내린 많은 비로 하천 수위가 높아져 안전사고가 우려되므로 \r\n하천 출입...   

                                          clean_text agency     region  \
0  밤사이 내린 눈과 비로 도로와 인도의 결빙이 우려되오니 감속운행, 대중교통 이용, ...    경주시  경상북도 경주시    
1  상주지역에 새벽까지 내린 눈으로 도로결빙이 우려되오니 차량운행시 감속 등 안전운전하...    상주시  경상북도 상주시    
2  밤새 내린 눈과 낮은기온으로 인하여 도로 결빙이 우려되오니 차량운행(감속운행, 차간...    아산시  충청남도 아산시    
3  북구에서 실종된 김균석씨(남,60세)를 찾습니다 -150cm,85kg,비만체격,흰색...  광주경찰청  광주광역시 북구    
4  전일부터 내린 많은 비로 하천 수위가 높아져 안전사고가 우려되므로 하천 출입 및 논...   경상남도   경상남도 전체    

  disaster_type alert_type  has_url  url_count  text_length  \
0            대설       안전안내        0          0 

In [ ]:
import pandas as pd
import re

# ==========================================
# 1. 데이터 불러오기
# ==========================================

df = pd.read_csv(
    "/content/drive/MyDrive/source_documents.csv"
)

print("원본 데이터 개수:", len(df))

# ==========================================
# 2. 중복 제거
# ==========================================

df = df.drop_duplicates(subset=["text_hash"])

# ==========================================
# 3. raw_text 없는 데이터 제거
# ==========================================

df = df.dropna(subset=["raw_text"])

print("중복 제거 후:", len(df))

# ==========================================
# 4. 텍스트 정제
# ==========================================

def clean_text(text):

    text = str(text)

    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    return text

df["clean_text"] = df["raw_text"].apply(
    clean_text
)

# ==========================================
# 5. 재난 유형 추출
# ==========================================

disaster_keywords = [
    "화재",
    "폭염",
    "호우",
    "태풍",
    "수난사고",
    "교통사고",
    "추락사고",
    "산악사고",
    "폭발사고",
    "폭발",
    "산불",
    "가스누출",
    "정전사고",
    "정전",
    "붕괴사고",
    "붕괴",
    "열차사고",
    "물놀이사고",
    "벌쏘임",
    "안전사고"
]

def find_disaster(text):

    found = []

    for keyword in disaster_keywords:

        if keyword in text:
            found.append(keyword)

    if len(found) == 0:
        return "기타"

    return ",".join(list(set(found)))

df["disaster_type"] = (
    df["clean_text"]
    .apply(find_disaster)
)

# ==========================================
# 6. 지역 추출
# ==========================================

regions = [
    "서울",
    "부산",
    "대구",
    "인천",
    "광주",
    "대전",
    "울산",
    "세종",
    "경기",
    "강원",
    "충북",
    "충남",
    "전북",
    "전남",
    "경북",
    "경남",
    "제주"
]

def extract_region(text):

    found = []

    for region in regions:

        if region in text:
            found.append(region)

    if len(found) == 0:
        return "기타"

    return ",".join(list(set(found)))

df["region"] = (
    df["clean_text"]
    .apply(extract_region)
)

# ==========================================
# 7. 사망자 수 추출
# ==========================================

def extract_death(text):

    matches = re.findall(
        r'사망\s*(\d+)명',
        text
    )

    if len(matches) == 0:
        return 0

    return sum(
        map(int, matches)
    )

df["death_count"] = (
    df["clean_text"]
    .apply(extract_death)
)

# ==========================================
# 8. 부상자 수 추출
# ==========================================

def extract_injury(text):

    matches = re.findall(
        r'부상\s*(\d+)명',
        text
    )

    if len(matches) == 0:
        return 0

    return sum(
        map(int, matches)
    )

df["injury_count"] = (
    df["clean_text"]
    .apply(extract_injury)
)

# ==========================================
# 9. URL 개수 추출
# ==========================================

url_pattern = r'https?://[^\s]+'

df["url_count"] = (
    df["clean_text"]
    .apply(
        lambda x: len(
            re.findall(url_pattern, x)
        )
    )
)

# ==========================================
# 10. 문자 길이
# ==========================================

df["text_length"] = (
    df["clean_text"]
    .str.len()
)

# ==========================================
# 11. 위험도 점수(프로젝트용)
# ==========================================

df["risk_score"] = (
    df["death_count"] * 5
    + df["injury_count"] * 2
)

# ==========================================
# 12. 최종 데이터셋
# ==========================================

final_df = df[
    [
        "id",
        "report_date",
        "disaster_type",
        "region",
        "death_count",
        "injury_count",
        "risk_score",
        "text_length",
        "clean_text"
    ]
]

print(final_df.head())

# ==========================================
# 13. 저장
# ==========================================

save_path = (
    "/content/drive/MyDrive/REDRED/"
    "소방청일일상황보고.csv"
)

final_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 50)
print("전처리 완료")
print("저장 위치:", save_path)
print("=" * 50)

# ==========================================
# 14. 간단한 통계 확인
# ==========================================

print("\n재난 유형 TOP10")
print(
    final_df["disaster_type"]
    .value_counts()
    .head(10)
)

print("\n지역 TOP10")
print(
    final_df["region"]
    .value_counts()
    .head(10)
)

print("\n사망자 총합")
print(
    final_df["death_count"].sum()
)

print("\n부상자 총합")
print(
    final_df["injury_count"].sum()
)

원본 데이터 개수: 3201
중복 제거 후: 2967
     id report_date           disaster_type                   region  \
0  3557  2017-07-26  화재,추락사고,안전사고,물놀이사고,벌쏘임     인천,제주,경남,전남,서울,부산,충남   
1  3558  2017-07-27        호우,화재,교통사고,물놀이사고  인천,경기,경북,강원,전남,서울,부산,대구   
2  3559  2017-07-28    화재,교통사고,물놀이사고,호우,벌쏘임  인천,경기,경북,강원,전남,서울,부산,대구   
3  3560  2017-07-29     폭발,화재,수난사고,정전,물놀이사고     경기,대전,강원,부산,광주,전북,경북   
4  3561  2017-07-30  화재,교통사고,수난사고,물놀이사고,벌쏘임  인천,경기,대전,세종,전남,서울,부산,전북   

   death_count  injury_count  risk_score  text_length  \
0            1            42          89          866   
1            1             8          21          919   
2            1            52         109          952   
3            1             0           5          895   
4            1             5          15          874   

                                          clean_text  
0  소방 안전 활동 상황 총 괄 구 분 화 재 구 조 구 급 건수 사망 부상 피해액 (...  
1  119 소방안전 활동상황 2017.7.28.(금) 06:00 119종합상황실 총 괄...  
2  119 소방안전 활동상황 2017.7.28.(금

In [ ]:
import pandas as pd
import re
from urllib.parse import urlparse

REGION_PATTERN = re.compile(
    r'([가-힣]{2}\s?\S*(?:시|군|구)?\s?\S*(?:동|읍|면|리)?)\s*『'
)


def parse_region_from_block(block: str) -> dict:
    match = REGION_PATTERN.search(block)
    region_raw = match.group(1).strip() if match else None
    tokens = region_raw.split() if region_raw else []

    return {
        "sido": tokens[0] if len(tokens) > 0 else None,
        "sigungu": tokens[1] if len(tokens) > 1 else None,
        "emd": tokens[2] if len(tokens) > 2 else None,
    }


# =====================================================
# 데이터 로드
# =====================================================

sms = pd.read_csv("/content/drive/MyDrive/REDRED/disaster_sms_preprocessed.csv")

fire = pd.read_csv(
    "/content/drive/MyDrive/REDRED/소방청일일상황보고.csv"
)

# =====================================================
# URL 추출 패턴
# =====================================================

URL_PATTERN = re.compile(
    r"""
    (
        https?://[^\s\]\)]+
        |
        www\.[^\s\]\)]+
        |
        [a-zA-Z0-9-]+\.
        (?:com|net|org|kr|go\.kr|co\.kr|or\.kr|ac\.kr|la|ly|io|me|tv)
        (?:/[^\s\]\)]*)?
    )
    """,
    re.VERBOSE | re.IGNORECASE
)

SHORTENER_DOMAINS = {
    "bit.ly",
    "vo.la",
    "tinyurl.com",
    "goo.gl",
    "t.co",
    "ow.ly",
    "is.gd",
    "buff.ly",
    "cutt.ly",
    "rb.gy"
}

# =====================================================
# URL 추출
# =====================================================

def extract_urls_from_text(text):

    text = str(text)

    urls = URL_PATTERN.findall(text)

    return list(set(urls))


# =====================================================
# 도메인 추출
# =====================================================

def extract_domains(urls):

    domains = []

    for url in urls:

        if not url.startswith(
            ("http://", "https://")
        ):
            url = "http://" + url

        try:

            domain = (
                urlparse(url)
                .netloc
                .replace("www.", "")
                .lower()
            )

            domains.append(domain)

        except Exception:
            pass

    return list(set(domains))


# =====================================================
# SMS URL Feature 생성
# =====================================================

sms["detected_urls"] = (
    sms["clean_text"]
    .apply(extract_urls_from_text)
)

sms["url_count_fixed"] = (
    sms["detected_urls"]
    .apply(len)
)

sms["has_url_fixed"] = (
    sms["url_count_fixed"] > 0
).astype(int)

sms["url_domains"] = (
    sms["detected_urls"]
    .apply(extract_domains)
)

sms["has_shortened_url"] = (
    sms["url_domains"]
    .apply(
        lambda domains: int(
            any(
                d in SHORTENER_DOMAINS
                for d in domains
            )
        )
    )
)

# =====================================================
# RawOfficialAlert 확장용 컬럼
# 이미 있으면 유지
# 없으면 생성
# =====================================================

required_columns = [
    "source_agency",
    "emergency_level",
    "serial_number",
    "created_at_source"
]

for col in required_columns:

    if col not in sms.columns:

        sms[col] = None

# =====================================================
# 저장
# Pass-through 유지
# =====================================================

sms.to_csv(
    "/content/drive/MyDrive/REDRED/sms_url_enriched.csv",
    encoding="utf-8-sig",
    index=False
)

print("sms_url_enriched.csv 생성 완료")

# =====================================================
# 사건 단위 분해
# =====================================================

event_pattern = r'❍\s*\((.*?)\)'

event_rows = []


def extract_count(pattern, text):

    nums = re.findall(
        pattern,
        str(text)
    )

    if not nums:
        return 0

    total = 0

    for n in nums:

        try:
            total += int(n)

        except Exception:
            pass

    return total


# =====================================================
# 사건 분해
# =====================================================

for _, row in fire.iterrows():

    raw_text = str(
        row["clean_text"]
    )

    matches = list(
        re.finditer(
            event_pattern,
            raw_text
        )
    )

    if len(matches) == 0:
        continue

    for i, match in enumerate(matches):

        start = match.start()

        end = (
            matches[i + 1].start()
            if i < len(matches) - 1
            else len(raw_text)
        )

        block = raw_text[start:end]

        category = match.group(1)

        region_info = parse_region_from_block(
            block
        )

        event_rows.append({

            # 원본 유지
            "report_id":
                row["id"],

            "report_date":
                row["report_date"],

            # 신규
            "category":
                category,

            "sido":
                region_info.get("sido"),

            "sigungu":
                region_info.get("sigungu"),

            "emd":
                region_info.get("emd"),

            "event_text":
                block,

            "death":
                extract_count(
                    r"사망\s*(\d+)명",
                    block
                ),

            "severe":
                extract_count(
                    r"중상\s*(\d+)명",
                    block
                ),

            "minor":
                extract_count(
                    r"경상\s*(\d+)명",
                    block
                )
        })

# =====================================================
# 사건 데이터 생성
# =====================================================

events = pd.DataFrame(
    event_rows
)

events.to_csv(
    "/content/drive/MyDrive/REDRED/fire_events.csv",
    encoding="utf-8-sig",
    index=False
)

print("fire_events.csv 생성 완료")
print("총 사건 수 :", len(events))
print("고유 report_id 수 :", events["report_id"].nunique())

print("모든 작업 완료")


sms_url_enriched.csv 생성 완료
fire_events.csv 생성 완료
총 사건 수 : 17854
고유 report_id 수 : 2967
모든 작업 완료
